# Cell-Akamai companion 80 — Workshop Module 8: Tune and Evaluate

**Executed output is produced only when YOU run these cells on Akamai JupyterLab.**

This companion supplements the upstream [`08_tune_and_evaluate.ipynb`](https://github.com/akamai-developers/akamai-workshop-ai-inference/blob/main/08_tune_and_evaluate/08_tune_and_evaluate.ipynb), retaining its **baseline → one flag → rollout → identical workload → compare** design. It adds server-usage-verified tokens/s and failure receipts because upstream `common/loadtest.py` counts streamed chunks, not tokenizer tokens.

Safety: no mutation until the explicitly labeled authorization cell is changed from `False` to `True`. Run Module 0 prerequisites first; do not execute these while Claude or Cursor is benchmarking the same GPU.

**Prerequisite:** active vLLM endpoint and authorized namespace-scoped kubeconfig. Models/precision must be verified, never inferred just from a folder label.

In [ ]:
import sys, os, json, time
from pathlib import Path
# The ZIP folder has notebooks/ one level below its helper.
P = Path.cwd().resolve()
PACKAGE = next((q for q in (P, *P.parents) if (q/'cell_akamai_handoff.py').is_file()), None)
assert PACKAGE, 'Open notebook in extracted ZIP folder/notebooks, or add ZIP folder to sys.path'
sys.path.insert(0,str(PACKAGE))
import cell_akamai_handoff as cak
print('Package:',PACKAGE)
print('Upstream:',cak.UPSTREAM_SHA)
print('This notebook makes NO calls until preflight below.')

## 1. Preflight — actual current serving configuration

Read live `/v1/models` and the current namespace's Kubernetes `deployment/vllm`; fail closed if they disagree. The saved `preflight.json` includes no credentials. Ensure you have completed upstream workshop Module 0 and that no other benchmark is active.

In [ ]:
ENV=cak.preflight()
print(json.dumps({k:v for k,v in ENV.items() if k!='deployment'},indent=2))
print('Live serving args:',ENV['deployment']['args'])
assert '--max-num-seqs=' in ' '.join(ENV['deployment']['args']), 'Tuning flag absent'
assert ENV['served_model']==ENV['deployment']['model_flag']

## 2. Execute frozen baseline (Module 8 before)

By default 4 bounded concurrency cells × 16 requests each. All prompts are fixed, generated identically in the after epoch. The recorded metric `aggregate_completion_tokens_per_s` is **sum of server-reported completion tokens divided by batch wall seconds**, not streamed chunks per second. `TTFC` is not measured; p95 is request wall latency. Running these calls creates actual GPU load.

In [ ]:
LEVELS=(1,4,8,16)
REQUESTS_PER_LEVEL=16
MAX_COMPLETION_TOKENS=96
M08_BEFORE=cak.load_epoch(ENV,'before',levels=LEVELS,per_level=REQUESTS_PER_LEVEL,max_tokens=MAX_COMPLETION_TOKENS)
print('Before evidence:',M08_BEFORE)
print(json.dumps(cak.read_json(M08_BEFORE/'summary.json')['cells'],indent=2))
M08_QUALITY_BEFORE=cak.quality_epoch(ENV,'before',n_items=8)
print('Quality BEFORE:',json.dumps(cak.score_quality(M08_QUALITY_BEFORE),indent=2))

## 3. Preview one-flag tuning change (read-only)

Module 8 requires an actual one-knob change for a measured comparison. This package restricts the intervention to `--max-num-seqs` while holding the served model, image, all other args and workload fixed. It never edits the upstream workshop files. Review the intended change and obtain namespace authorization before running the following mutation cell.

In [ ]:
NEW_MAX_NUM_SEQS=64
PLAN=cak.apply_tune(ENV,M08_BEFORE,new_value=NEW_MAX_NUM_SEQS,authorize=False)
print(json.dumps(PLAN,indent=2))

## 4. Explicit operator authorization and rollout

**This is a live mutation.** Review `tune_plan.json` and change `AUTHORIZE_ONE_FLAG_ROLLOUT` to `True` **only if you own this namespace and are allowed to restart your vLLM deployment.** It sends one guarded Kubernetes JSON patch and waits up to 12 minutes for rollout. The original args remain in the baseline receipt for manual restoration. No automatic rollback: restoration is a separate authorized change.

In [ ]:
AUTHORIZE_ONE_FLAG_ROLLOUT=False  # SET True ONLY WHEN YOU AUTHORIZE THE GPU RESTART
if AUTHORIZE_ONE_FLAG_ROLLOUT:
    cak.apply_tune(ENV,M08_BEFORE,new_value=NEW_MAX_NUM_SEQS,authorize=True)
    print('ROLL-OUT APPLIED: vLLM verified ready with one changed flag.')
else:
    print('DEFERRED: no deployment mutation; the after measurement will not run.')

## 5. After epoch + measured Module 8 comparison

This will only run if the guarded rollout receipt exists. The validation rejects config drift, model swaps, workload mismatch, and changes to more than one flag. Throughput comparisons are **descriptive**, not proof of causal improvement; temporal interference and the separate Module 9 quality check are retained.

In [ ]:
M08_AFTER=None
if not (M08_BEFORE/'tune_actual.json').is_file():
    print('NOT_TESTED: authorized one-flag rollout not executed. Module 8 before-only is still exportable.')
else:
    ENV_AFTER=cak.preflight()
    if ENV_AFTER['served_model'] != ENV['served_model']:
        raise RuntimeError('Served model changed; comparison INVALID')
    M08_AFTER=cak.load_epoch(ENV_AFTER,'after',levels=LEVELS,per_level=REQUESTS_PER_LEVEL,max_tokens=MAX_COMPLETION_TOKENS)
    result=cak.compare_tune(M08_BEFORE,M08_AFTER)
    print(json.dumps(result,indent=2))
    M08_QUALITY_AFTER=cak.quality_epoch(ENV_AFTER,'after',n_items=8)
    print('Quality AFTER:',json.dumps(cak.score_quality(M08_QUALITY_AFTER),indent=2))
    print('Quality pairing:',json.dumps(cak.compare_quality(M08_QUALITY_BEFORE,M08_QUALITY_AFTER),indent=2))

## 6. Evidence boundaries and handoff

- Keep both evidence directories; do not overwrite earlier experiments.
- A request's `completion_tokens` derives from the endpoint's usage field. Unknown usage stays null.
- Prometheus `generation_tokens_total` delta is checked against the measured token sum. Mismatches indicate external load, reset, or metric ambiguity and cannot be promoted as an isolated benchmark.
- `tokens/s²` is deliberately `NOT_COMPUTED` for concurrent cells; a sequential inter-request derivative is a distinct measurement.
- Module 9 companion `81_m09_agent_and_daisy.ipynb` evaluates deployment and independent correctness. Then open `82_verify_and_export.ipynb` to download one ZIP of your runs.